In [1]:
import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.models import Model

DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"

train_dir = os.path.join(DATASET_PATH, "train")
val_dir   = os.path.join(DATASET_PATH, "val")
test_dir  = os.path.join(DATASET_PATH, "test")

CLASSES = ["edible", "poisonous"]

IMG_SIZE = 224
BATCH_SIZE = 32
PRUNE_STAGES = [0.2, 0.3, 0.4]
EPOCHS_PER_STAGE = 10
 # ITERATIVE

def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    image = preprocess_input(image)
    return image

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

attention_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_test_gen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_gen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

attention_data = attention_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

BASE_MODEL_PATH = r"D:\Project Jupyter\FINAL QUANTIZATION FIX\HASIL MODELBASE\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"



Found 2256 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
Found 232 images belonging to 2 classes.
Found 2256 images belonging to 2 classes.


In [2]:
model = tf.keras.models.load_model(BASE_MODEL_PATH, compile=False)
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3large (Functiona  (None, 7, 7, 960)        2996352   
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 960)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               123008    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 258       
                                                                 
Total params: 3,119,618
Trainable params: 123,266
Non-tr

In [3]:
backbone = model.layers[0]

In [4]:
def extract_se_attention(backbone, dataset, max_batches=20):

    se_layers = [
        l for l in backbone.layers
        if "squeeze_excite" in l.name and isinstance(l, tf.keras.layers.Multiply)
    ]

    se_layer = se_layers[-1]  # SE terakhir (960 channel)

    att_model = Model(backbone.input, se_layer.output)

    attentions = []
    for i, (x, _) in enumerate(dataset):
        if i >= max_batches:
            break
        se_out = att_model(x, training=False)   # (B,1,1,960)
        att = tf.reduce_mean(se_out, axis=[0,1,2])
        attentions.append(att.numpy())

    return np.mean(np.stack(attentions), axis=0)


In [5]:
def build_channel_mask(attention, prune_ratio):
    C = attention.shape[0]
    keep = int(C * (1 - prune_ratio))

    threshold = np.sort(attention)[-keep]
    mask = (attention >= threshold).astype(np.float32)

    print(f"Keep channels: {int(mask.sum())}/{C}")
    return mask


In [6]:
class ChannelMask(tf.keras.layers.Layer):
    def __init__(self, mask, **kwargs):
        super().__init__(trainable=False, **kwargs)
        self.mask = tf.reshape(
            tf.constant(mask, dtype=tf.float32),
            [1, 1, 1, -1]
        )

    def call(self, x):
        return x * self.mask


In [7]:
current_mask = np.ones(960, dtype=np.float32)

inputs = tf.keras.Input(shape=(224,224,3))
x = backbone(inputs, training=False)

for stage, ratio in enumerate(PRUNE_STAGES):
    print(f"\n===== ITERATIVE PRUNE {int(ratio*100)}% =====")

    # 1️⃣ hitung attention terbaru
    attention = extract_se_attention(backbone, attention_data)

    # 2️⃣ build mask baru
    stage_mask = build_channel_mask(attention, ratio)

    # 3️⃣ mask bersifat kumulatif
    current_mask = current_mask * stage_mask

    # 4️⃣ apply mask
    x = ChannelMask(current_mask, name=f"channel_mask_stage_{stage+1}")(x)



===== ITERATIVE PRUNE 20% =====
Keep channels: 768/960

===== ITERATIVE PRUNE 30% =====
Keep channels: 672/960

===== ITERATIVE PRUNE 40% =====
Keep channels: 576/960


In [8]:
x = model.layers[1](x)  # GAP
x = model.layers[2](x)  # Dense 128
x = model.layers[3](x)  # Dropout
outputs = model.layers[4](x)  # Dense 2

pruned_model = Model(inputs, outputs)
pruned_model.summary()


Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 MobilenetV3large (Functiona  (None, 7, 7, 960)        2996352   
 l)                                                              
                                                                 
 channel_mask_stage_1 (Chann  (None, 7, 7, 960)        0         
 elMask)                                                         
                                                                 
 channel_mask_stage_2 (Chann  (None, 7, 7, 960)        0         
 elMask)                                                         
                                                                 
 channel_mask_stage_3 (Chann  (None, 7, 7, 960)        0         
 elMask)                                                   

In [9]:
import time

pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

start_time = time.time()

for stage, ratio in enumerate(PRUNE_STAGES):
    print(f"\n--- Fine-tuning after {int(ratio*100)}% prune ---")

    pruned_model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS_PER_STAGE,
        callbacks=[early_stop],
        verbose=1
    )

end_time = time.time()

training_time = end_time - start_time

print(f"\nTotal Training Time : {training_time:.2f} seconds")
print(f"Total Training Time : {training_time/60:.2f} minutes")
print(f"Avg Time / Epoch    : {training_time/(EPOCHS_PER_STAGE*len(PRUNE_STAGES)):.2f} seconds")



--- Fine-tuning after 20% prune ---
Epoch 1/10
71/71 [==============================] - 177s 2s/step - loss: 0.3223 - accuracy: 0.8635 - val_loss: 0.3217 - val_accuracy: 0.8617
Epoch 2/10
71/71 [==============================] - 249s 4s/step - loss: 0.2964 - accuracy: 0.8746 - val_loss: 0.3131 - val_accuracy: 0.8688
Epoch 3/10
71/71 [==============================] - 295s 4s/step - loss: 0.2823 - accuracy: 0.8812 - val_loss: 0.3032 - val_accuracy: 0.8901
Epoch 4/10
71/71 [==============================] - 306s 4s/step - loss: 0.2613 - accuracy: 0.8923 - val_loss: 0.2865 - val_accuracy: 0.8972
Epoch 5/10
71/71 [==============================] - 277s 4s/step - loss: 0.2448 - accuracy: 0.9051 - val_loss: 0.2842 - val_accuracy: 0.8936
Epoch 6/10
71/71 [==============================] - 268s 4s/step - loss: 0.2430 - accuracy: 0.9060 - val_loss: 0.2794 - val_accuracy: 0.8936
Epoch 7/10
71/71 [==============================] - 349s 5s/step - loss: 0.2393 - accuracy: 0.9056 - val_loss: 0.2755

In [10]:
PRUNED_MODEL_PATH = r"D:\mobilenetv3_PRUNED_20_30_40FINALAGP"
pruned_model.save(
    PRUNED_MODEL_PATH,
    include_optimizer=False
)


INFO:tensorflow:Assets written to: D:\mobilenetv3_se_attention_pruned_final20_30_40FINALAGP10\assets


INFO:tensorflow:Assets written to: D:\mobilenetv3_se_attention_pruned_final20_30_40FINALAGP10\assets


In [11]:
import os

def get_model_size_mb(model_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)


In [12]:
base_model_size = get_model_size_mb(BASE_MODEL_PATH)
pruned_model_size = get_model_size_mb(PRUNED_MODEL_PATH)

print(f"Ukuran Baseline Model : {base_model_size:.2f} MB")
print(f"Ukuran Pruned Model   : {pruned_model_size:.2f} MB")

compression_ratio = base_model_size / pruned_model_size
size_reduction = (1 - (pruned_model_size / base_model_size)) * 100

print(f"Compression Ratio : {compression_ratio:.2f}x")
print(f"Size Reduction    : {size_reduction:.2f}%")


Ukuran Baseline Model : 17.87 MB
Ukuran Pruned Model   : 16.85 MB
Compression Ratio : 1.06x
Size Reduction    : 5.68%


In [13]:

from sklearn.metrics import classification_report, confusion_matrix


In [14]:
test_data.reset()


In [15]:
predictions = pruned_model.predict(test_data, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes


8/8 [==============================] - 20s 2s/step


In [16]:
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=CLASSES
))



Classification Report:
              precision    recall  f1-score   support

      edible       0.96      0.97      0.96       120
   poisonous       0.96      0.96      0.96       112

    accuracy                           0.96       232
   macro avg       0.96      0.96      0.96       232
weighted avg       0.96      0.96      0.96       232



In [17]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))



Confusion Matrix:
[[116   4]
 [  5 107]]


In [17]:
import time
import numpy as np

def benchmark_inference(model, data_gen, warmup=3, runs=10):
    """
    model      : final pruned model (tanpa pruning wrapper)
    data_gen   : test_data (shuffle=False)
    """

    # Ambil 1 batch
    x_batch, _ = next(data_gen)

    # Warm-up (penting)
    for _ in range(warmup):
        _ = pruned_model.predict(x_batch, verbose=0)

    times = []

    for _ in range(runs):
        start = time.time()
        _ = pruned_model.predict(x_batch, verbose=0)
        end = time.time()
        times.append(end - start)

    avg_time = np.mean(times)
    per_image_time = avg_time / x_batch.shape[0]

    print("\nInference Benchmark (CPU)")
    print(f"Batch size         : {x_batch.shape[0]}")
    print(f"Avg batch time     : {avg_time:.4f} seconds")
    print(f"Avg per image time : {per_image_time:.6f} seconds")

    return avg_time, per_image_time


In [25]:
benchmark_inference(base_model, test_data)


Inference Benchmark (CPU)
Batch size         : 32
Avg batch time     : 1.0106 seconds
Avg per image time : 0.031582 seconds


(1.0106214523315429, 0.031581920385360715)